# CPE 342 Term Project — Task 4: Game Title Detection (Image Classification)
**Course:** CPE 342 Machine Learning (Semester 1/2026) — KMUTT  
**System Component:** Karena Thailand Player Intelligence System — Automated Support & Screenshot Verification  
**Task Category:** Multi-Class Image Classification (5 Game Titles, Widescreen RGB Gameplay Screenshots)  
**Primary Evaluation Metric:** **Macro $F_1$ Score**  

### Official Competition Rule Compliance
* **Data Usage Policy (Rule 2):** Uses exclusively the provided screenshot dataset (`train.csv` + `train/`); Data Augmentation (`RandomHorizontalFlip`, `ColorJitter`) is applied strictly to the training set. Zero test-set pseudo-labeling or fine-tuning.
* **Model Development (Rule 3):** **Pre-trained vision models explicitly allowed** (`convnext_tiny`, `swin_tiny`, `efficientnet_b0`, `resnet50`, `vit`). Zero AutoML frameworks.
* **Portable GPU Execution:** Ready for **Google Colab GPU (T4 / L4 / A100)**, **Kaggle GPU (T4 x2)**, and **Apple Silicon MPS**.

## 1. Environment Setup, Hardware GPU Detection & Automatic Dataset Downloader
Supports seamless execution on **Google Colab**, **Kaggle Notebooks**, and **Local Machines**:
* If the image archive has not yet been extracted into `task4/data/`, `ensure_task4_dataset()` automatically downloads `task4_image_classification.zip` (579 MB) from the official Google Drive ID (`1C8csylbBo3Oab24ODGKPVom3_SDUJ_Qw`) specified in `task4/data/README.md` and extracts it in place.

In [ ]:
import os
import sys
import zipfile
import warnings
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

warnings.filterwarnings("ignore")

for pkg in ["timm", "gdown", "scikit-learn", "tqdm"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        if "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
            os.system(f"pip install -q {pkg}")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BASE_DIR = Path.cwd()
if (BASE_DIR / "project" / "task4").exists():
    TASK_DIR = BASE_DIR / "project" / "task4"
elif (BASE_DIR / "task4").exists():
    TASK_DIR = BASE_DIR / "task4"
else:
    TASK_DIR = BASE_DIR

DATA_DIR = TASK_DIR / "data"
MODELS_DIR = TASK_DIR / "models"
PREDICTIONS_DIR = TASK_DIR / "predictions"
ASSETS_DIR = TASK_DIR / "assets"

for d in [DATA_DIR, MODELS_DIR, PREDICTIONS_DIR, ASSETS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

GDRIVE_FILE_ID = "1C8csylbBo3Oab24ODGKPVom3_SDUJ_Qw"

def ensure_task4_dataset(data_dir: Path):
    train_csv_candidates = list(data_dir.rglob("train.csv"))
    if train_csv_candidates:
        print(f"Found Task 4 dataset at: {train_csv_candidates[0].parent}")
        return train_csv_candidates[0].parent
    
    zip_path = data_dir / "task4_image_classification.zip"
    if not zip_path.exists():
        print("Downloading task4_image_classification.zip (579 MB) from official Google Drive...")
        import gdown
        gdown.download(id=GDRIVE_FILE_ID, output=str(zip_path), quiet=False)
    
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(data_dir)
        train_csv_candidates = list(data_dir.rglob("train.csv"))
        if train_csv_candidates:
            return train_csv_candidates[0].parent
    return data_dir

print(f"Compute Device : {DEVICE}")
print(f"Task 4 Dir     : {TASK_DIR}")

## 2. Dataset Loading, Label Verification & Rectangular Widescreen Pipeline ($128 \times 256$)
* **Aspect Ratio Preservation:** Native gameplay screenshots have a widescreen rectangular resolution ($230 \times 120$ / $200 \times 100$ pixels, aspect ratio $\approx 2:1$). Forcing a square $224 \times 224$ resize distorts HUD minimaps, health bars, and typography. We preserve the native horizontal aspect ratio by resizing to **$128 \times 256$ pixels**.
* **Dual Label Output:** Saves both the integer class ID (`0`–`4`, matching `sample_submission.csv`) and the string game label (`7-11K`, `fiveN`, `freefried`, `pube`, `roblock`).

In [ ]:
DATASET_ROOT = ensure_task4_dataset(DATA_DIR)

train_csv_path = DATASET_ROOT / "train.csv"
test_csv_path = DATASET_ROOT / "test.csv"

if train_csv_path.exists() and test_csv_path.exists():
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    print(f"Loaded Task 4 train.csv: {train_df.shape} | test.csv: {test_df.shape}")
    display(train_df.head())
else:
    print("NOTE: train.csv / test.csv not yet extracted locally. Run ensure_task4_dataset() on Colab/Kaggle GPU or local machine with internet to auto-download.")

## 3. PyTorch Dataset, Data Augmentation & Pretrained Backbone (`ConvNeXt-Tiny` / `EfficientNet-B0`)
* Applies **RandomHorizontalFlip** and **ColorJitter** strictly on training folds to generalize across diverse lighting, maps, and HUD overlays.
* Trains with **Label Smoothing Cross-Entropy Loss**, **AdamW**, and **CosineAnnealingLR**.

In [ ]:
IMG_SIZE = (128, 256)  # (Height, Width) preserving widescreen aspect ratio
BATCH_SIZE = 64
EPOCHS = 8
LR = 3e-4
NUM_CLASSES = 5

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class KarenaScreenshotDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: Path, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
        self.id_col = "id" if "id" in df.columns else df.columns[0]
        self.target_col = None if is_test else ([c for c in ["class", "label", "game_title", "target"] if c in df.columns] or [df.columns[-1]])[0]

    def __len__(self):
        return len(self.df)

    def _resolve_img_path(self, img_id: str) -> Path:
        img_id_str = str(img_id)
        for ext in ["", ".jpg", ".png", ".jpeg"]:
            cand = self.img_dir / f"{img_id_str}{ext}"
            if cand.exists():
                return cand
        raise FileNotFoundError(f"Image not found for ID: {img_id_str} in {self.img_dir}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_img_path(row[self.id_col])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, str(row[self.id_col])
        return img, int(row[self.target_col])

def build_pretrained_model(backbone_name="convnext_tiny", num_classes=5):
    try:
        import timm
        return timm.create_model(backbone_name, pretrained=True, num_classes=num_classes)
    except Exception:
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model

## 4. Stratified Cross-Validation Training Loop & Test-Time Augmentation (TTA)
Uses **2x Test-Time Augmentation (Original + Horizontal Flip)** during validation and test inference to boost **Macro $F_1$** across all 25,889 test screenshots, saving predictions to `task4/predictions/test_predictions.csv`.

In [ ]:
def train_and_evaluate_task4(backbone="convnext_tiny", n_splits=5, folds_to_run=5):
    if not (train_csv_path.exists() and test_csv_path.exists()):
        print("Dataset files not found yet. Once downloaded via ensure_task4_dataset(), run this function to train.")
        return
    
    train_img_dir = DATASET_ROOT / "train"
    test_img_dir = DATASET_ROOT / "test"
    
    target_col = ([c for c in ["class", "label", "game_title", "target"] if c in train_df.columns] or [train_df.columns[-1]])[0]
    y_all = train_df[target_col].values
    
    # Map labels if string
    if y_all.dtype == object:
        unique_classes = sorted(pd.Series(y_all).unique())
        label_to_idx = {c: i for i, c in enumerate(unique_classes)}
        idx_to_label = {i: c for c, i in label_to_idx.items()}
        train_df["target_idx"] = train_df[target_col].map(label_to_idx)
    else:
        idx_to_label = {0: "7-11K", 1: "fiveN", 2: "freefried", 3: "pube", 4: "roblock"}
        train_df["target_idx"] = train_df[target_col].astype(int)
        
    y_idx = train_df["target_idx"].values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    oof_probs = np.zeros((len(train_df), NUM_CLASSES))
    test_probs = np.zeros((len(test_df), NUM_CLASSES))
    
    test_ds = KarenaScreenshotDataset(test_df, test_img_dir, transform=val_transform, is_test=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
    
    for fold, (trn_idx, val_idx) in enumerate(skf.split(train_df, y_idx), 1):
        if fold > folds_to_run:
            break
        print(f"\n--- Training Fold {fold}/{folds_to_run} ({backbone}) on {DEVICE} ---")
        trn_ds = KarenaScreenshotDataset(train_df.iloc[trn_idx], train_img_dir, transform=train_transform)
        val_ds = KarenaScreenshotDataset(train_df.iloc[val_idx], train_img_dir, transform=val_transform)
        
        trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
        
        model = build_pretrained_model(backbone, NUM_CLASSES).to(DEVICE)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        
        best_val_f1 = -1.0
        best_val_probs = None
        ckpt_path = MODELS_DIR / f"{backbone}_fold{fold}.pt"
        
        for epoch in range(1, EPOCHS + 1):
            model.train()
            for imgs, labels in tqdm(trn_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                logits = model(imgs)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()
            scheduler.step()
            
            # Validation with 2x TTA (Original + Horizontal Flip)
            model.eval()
            val_preds_list = []
            with torch.no_grad():
                for imgs, _ in val_loader:
                    imgs = imgs.to(DEVICE)
                    p1 = torch.softmax(model(imgs), dim=1)
                    p2 = torch.softmax(model(torch.flip(imgs, dims=[3])), dim=1)
                    val_preds_list.append(((p1 + p2) * 0.5).cpu().numpy())
            val_probs = np.vstack(val_preds_list)
            val_f1 = f1_score(y_idx[val_idx], val_probs.argmax(axis=1), average="macro")
            print(f"  Epoch {epoch}/{EPOCHS} | Val Macro F1 (TTA): {val_f1:.4f}")
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_val_probs = val_probs
                torch.save(model.state_dict(), ckpt_path)
                
        oof_probs[val_idx] = best_val_probs
        
        # Test Inference with best fold checkpoint + 2x TTA
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.eval()
        fold_test_preds = []
        with torch.no_grad():
            for imgs, _ in tqdm(test_loader, desc=f"Test Inference Fold {fold}", leave=False):
                imgs = imgs.to(DEVICE)
                p1 = torch.softmax(model(imgs), dim=1)
                p2 = torch.softmax(model(torch.flip(imgs, dims=[3])), dim=1)
                fold_test_preds.append(((p1 + p2) * 0.5).cpu().numpy())
        test_probs += np.vstack(fold_test_preds) / folds_to_run
        
    # Save Confusion Matrix & Test Predictions
    pred_classes = test_probs.argmax(axis=1)
    test_out = pd.DataFrame({
        "id": test_df[test_ds.id_col],
        "task4": pred_classes.astype(int),
        "task4_label_name": [idx_to_label.get(int(i), str(i)) for i in pred_classes],
    })
    for c in range(NUM_CLASSES):
        test_out[f"prob_class_{c}"] = test_probs[:, c]
    test_out.to_csv(PREDICTIONS_DIR / "test_predictions.csv", index=False)
    print(f"Saved Task 4 predictions to: {PREDICTIONS_DIR / 'test_predictions.csv'}")

# Ready to execute when dataset archive is extracted:
if train_csv_path.exists() and test_csv_path.exists():
    train_and_evaluate_task4(backbone="convnext_tiny", n_splits=5, folds_to_run=5)

## 5. Executive Conclusion & Architecture Summary
* **Aspect Ratio Preservation:** Resizing screenshots to a rectangular $128 \times 256$ widescreen tensor preserves the native $\approx 1.92:1$ aspect ratio ($230 \times 120$), preventing geometric distortion of HUD minimaps, health bars, and typography.
* **Transfer Learning & TTA:** Combining ImageNet-pretrained backbones (`convnext_tiny`, `swin_tiny`, `efficientnet_b0`) with Label Smoothing Cross-Entropy (`0.05`), Cosine Annealing learning rate scheduling, and 2x Test-Time Augmentation maximizes Macro $F_1$ across all 5 Karena game titles.